# 🎯 Generate Targets — Interpolation Experiment

**Stage 1** of the DPS interpolation experiment.

Generates:
- One canonical **masculine man** portrait → VAE latent duplicated 50×
- One canonical **feminine woman** portrait → VAE latent duplicated 50×  
- **100-step linear interpolation** in VAE latent space between the two
- Decodes all 100 frames to PNG for visual inspection
- Saves PCA visualisations (latent-space + CLIP perceptual)
- Writes `metadata.json` and fitted PCA models consumed by `run_interpolation_dps.py`

**All outputs are saved under `OUTPUT_DIR`** so Stage 2 can load them without re-running.

## ⚙️ Configuration — edit here before running

In [ ]:
# ── Output directory ──────────────────────────────────────────────────────────
OUTPUT_DIR = "SD_cond_SD_controlnet/output/interpolation_experiment"

# ── Experiment parameters ─────────────────────────────────────────────────────
N_COPIES    = 50    # how many identical copies to stack per canonical latent
N_INTERP    = 100   # number of interpolation steps (including endpoints)

# ── Generation parameters ─────────────────────────────────────────────────────
CONTROLNET_SCALE = 0.4   # soft conditioning — let the prompt dominate
MAN_SEED         = 0     # fixed seed → deterministic canonical man
WOMAN_SEED       = 1     # fixed seed → deterministic canonical woman
GLOBAL_SEED      = 42

# ── Model IDs (HuggingFace) ───────────────────────────────────────────────────
CONTROLNET_MODEL_ID = "xinsir/controlnet-scribble-sdxl-1.0"
SPRINTER_MODEL_ID   = "stabilityai/sdxl-turbo"
ARCHITECT_MODEL_ID  = "stabilityai/stable-diffusion-xl-base-1.0"

# ── Prompts ───────────────────────────────────────────────────────────────────
MAN_PROMPT = (
    "a hyperrealistic studio portrait photograph of a very masculine man, "
    "strong jawline, short hair, formal attire, sharp features, "
    "professional photography, 8k"
)
WOMAN_PROMPT = (
    "a hyperrealistic studio portrait photograph of a very feminine woman, "
    "long hair, soft features, elegant attire, beautiful, "
    "professional photography, 8k"
)

print(f"OUTPUT_DIR : {OUTPUT_DIR}")
print(f"N_COPIES   : {N_COPIES}")
print(f"N_INTERP   : {N_INTERP}")
print(f"MAN_SEED   : {MAN_SEED}   WOMAN_SEED: {WOMAN_SEED}")

## 1 · Environment Setup

In [ ]:
# Install any missing packages (safe to re-run)
import subprocess, sys
pkgs = ["diffusers", "transformers", "accelerate", "controlnet_aux",
        "scikit-learn", "tqdm", "peft"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)
print("✅ Packages ready.")

In [ ]:
import os, sys

# ── Locate the SD_cond_SD_controlnet module ───────────────────────────────────
# Works whether the notebook is in the repo root or in SD_cond_SD_controlnet/
NOTEBOOK_DIR = os.path.abspath("")
CANDIDATES = [
    os.path.join(NOTEBOOK_DIR, "SD_cond_SD_controlnet"),
    os.path.join(NOTEBOOK_DIR, "..", "SD_cond_SD_controlnet"),
    NOTEBOOK_DIR,  # notebook IS inside SD_cond_SD_controlnet/
]
MODULE_DIR = None
for c in CANDIDATES:
    c = os.path.normpath(c)
    if os.path.isfile(os.path.join(c, "models.py")):
        MODULE_DIR = c
        break

assert MODULE_DIR is not None, (
    "Could not find SD_cond_SD_controlnet/models.py. "
    "Place this notebook in the repo root or inside SD_cond_SD_controlnet/."
)
if MODULE_DIR not in sys.path:
    sys.path.insert(0, MODULE_DIR)

print(f"✅ Module dir: {MODULE_DIR}")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Output dir: {os.path.abspath(OUTPUT_DIR)}")

## 2 · Imports

In [ ]:
import json, pickle
import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from IPython.display import display as ipy_display
from sklearn.decomposition import PCA
from tqdm.notebook import tqdm

from models      import load_models
from clip_utils  import load_clip_model, encode_images_clip
from image_utils import build_base_image, sobel_proxy

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Device: {device}")

torch.manual_seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

## 3 · Helper Functions

In [ ]:
def generate_one(pipe, prompt, cond_pil, cn_scale, seed):
    """
    Generate a single image with a fixed seed.
    Returns (PIL image, latent [4,64,64] float32, pipeline-scaled).
    The latent is already multiplied by vae.config.scaling_factor.
    """
    generator = torch.Generator(device=pipe.device).manual_seed(seed)
    pipe.vae.to(dtype=torch.float16)
    with torch.no_grad():
        result = pipe(
            prompt=[prompt],
            image=[cond_pil],
            num_inference_steps=2,
            guidance_scale=0.0,
            controlnet_conditioning_scale=cn_scale,
            output_type="latent",
            return_dict=True,
            generator=generator,
        )
        lat = result.images  # [1, 4, 64, 64] fp16
        decoded = pipe.vae.decode(lat / pipe.vae.config.scaling_factor).sample
        decoded = torch.clamp((decoded.float() + 1.0) / 2.0, 0.0, 1.0)
        pil = TF.to_pil_image(decoded[0].cpu())
    pipe.vae.to(dtype=torch.float32)
    return pil, lat[0].float().cpu()  # PIL, [4,64,64]


def decode_latents(latents_batch, vae, batch_size=4):
    """
    Decode [N, 4, 64, 64] float32 pipeline-scaled latents → list of PIL images.
    """
    dev = next(vae.parameters()).device
    images = []
    vae.to(dtype=torch.float16)
    with torch.no_grad():
        for i in range(0, len(latents_batch), batch_size):
            batch = latents_batch[i:i + batch_size].to(dev).half()
            out = vae.decode(batch / vae.config.scaling_factor).sample
            out = torch.clamp((out.float() + 1.0) / 2.0, 0.0, 1.0)
            for j in range(out.shape[0]):
                images.append(TF.to_pil_image(out[j].cpu()))
    vae.to(dtype=torch.float32)
    return images


def encode_pil_to_clip(pil_list, clip_model, clip_processor, batch_size=8):
    """Encode list of PIL images → [N, 768] float32 CLIP embeddings."""
    all_embs = []
    clip_model.to(device)
    with torch.no_grad():
        for i in range(0, len(pil_list), batch_size):
            batch = pil_list[i:i + batch_size]
            tensors = torch.cat(
                [TF.to_tensor(img).unsqueeze(0) for img in batch], dim=0
            ).to(device)
            embs = encode_images_clip(tensors, clip_model, clip_processor)
            all_embs.append(embs.cpu())
    clip_model.to("cpu")
    return torch.cat(all_embs, dim=0).float()


def show_pil_row(images, titles=None, figsize_per=3):
    """Display a row of PIL images inline in the notebook."""
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(figsize_per * n, figsize_per))
    if n == 1:
        axes = [axes]
    for i, (ax, img) in enumerate(zip(axes, images)):
        ax.imshow(img)
        ax.axis("off")
        if titles:
            ax.set_title(titles[i], fontsize=8)
    plt.tight_layout()
    plt.show()


def plot_pca_interp(coords, n_steps, save_path, title):
    """Plot the interpolation path in 2D PCA space and display inline."""
    fig, ax = plt.subplots(figsize=(10, 5))
    sc = ax.scatter(
        coords[:, 0], coords[:, 1],
        c=np.arange(n_steps), cmap="RdBu_r",
        s=60, edgecolors="black", linewidths=0.3, zorder=3,
    )
    plt.colorbar(sc, ax=ax, label="Step  (0 = man,  N−1 = woman)")
    ax.plot(coords[:, 0], coords[:, 1], "k--", alpha=0.25, linewidth=1, zorder=2)
    ax.scatter(*coords[0],  s=250, c="royalblue", marker="*", zorder=5, label="Man   (α=0)")
    ax.scatter(*coords[-1], s=250, c="crimson",   marker="*", zorder=5, label="Woman (α=1)")
    for idx in range(0, n_steps, max(1, n_steps // 10)):
        ax.annotate(str(idx), coords[idx],
                    textcoords="offset points", xytext=(4, 4), fontsize=7, alpha=0.8)
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(save_path, dpi=130, bbox_inches="tight")
    plt.show()
    print(f"  ✅ Saved → {save_path}")

print("✅ Helpers defined.")

## 4 · Load Models

In [ ]:
print("Loading sprinter + ControlNet  (may download on first run)...")
_, sprinter = load_models(
    device,
    controlnet_model_id=CONTROLNET_MODEL_ID,
    sprinter_model_id=SPRINTER_MODEL_ID,
    architect_model_id=ARCHITECT_MODEL_ID,
)
print("Loading CLIP...")
clip_model, clip_processor = load_clip_model(device)
clip_model.to("cpu")  # keep off-GPU until needed
print("✅ Models loaded.")

## 5 · Build Base Scribble Conditioning

In [ ]:
base_image_pil, base_tensor = build_base_image(device)
with torch.no_grad():
    sobel_tensor = sobel_proxy(base_tensor, device)
    cond_pil = T.ToPILImage()(sobel_tensor.squeeze(0).cpu())

cond_pil.save(os.path.join(OUTPUT_DIR, "base_scribble.png"))

show_pil_row([base_image_pil, cond_pil],
             titles=["Base oval shape", "Sobel scribble (conditioning)"])
print("✅ base_scribble.png saved.")

## 6 · Generate the Two Canonical Portraits

In [ ]:
print(f"Generating canonical man  (seed={MAN_SEED})...")
man_pil, man_latent = generate_one(
    sprinter, MAN_PROMPT, cond_pil, CONTROLNET_SCALE, seed=MAN_SEED
)
man_pil.save(os.path.join(OUTPUT_DIR, "man_canonical.png"))
print(f"  Latent shape : {man_latent.shape}")
print(f"  Latent norm  : {man_latent.norm():.3f}")
show_pil_row([man_pil], titles=[f"Canonical man  (seed={MAN_SEED})"])

In [ ]:
print(f"Generating canonical woman (seed={WOMAN_SEED})...")
woman_pil, woman_latent = generate_one(
    sprinter, WOMAN_PROMPT, cond_pil, CONTROLNET_SCALE, seed=WOMAN_SEED
)
woman_pil.save(os.path.join(OUTPUT_DIR, "woman_canonical.png"))
print(f"  Latent shape : {woman_latent.shape}")
print(f"  Latent norm  : {woman_latent.norm():.3f}")
show_pil_row([woman_pil], titles=[f"Canonical woman (seed={WOMAN_SEED})"])

In [ ]:
# Distance stats between the two canonical latents
man_flat   = man_latent.reshape(-1).float()
woman_flat = woman_latent.reshape(-1).float()
cos_sim = torch.dot(man_flat / man_flat.norm(),
                    woman_flat / woman_flat.norm()).item()
l2_dist = (man_latent - woman_latent).norm().item()

print(f"Man ↔ Woman  cosine similarity (VAE latent) : {cos_sim:.4f}")
print(f"Man ↔ Woman  L2 distance       (VAE latent) : {l2_dist:.3f}")
print()
print("Side-by-side:")
show_pil_row([man_pil, woman_pil], titles=["Man (α=0)", "Woman (α=1)"])

## 7 · Duplicate Each Latent N_COPIES Times

In [ ]:
# [N_COPIES, 4, 64, 64] — all rows identical
man_latents   = man_latent.unsqueeze(0).expand(N_COPIES, -1, -1, -1).clone()
woman_latents = woman_latent.unsqueeze(0).expand(N_COPIES, -1, -1, -1).clone()

torch.save(man_latents,   os.path.join(OUTPUT_DIR, "man_vae_latents.pt"))
torch.save(woman_latents, os.path.join(OUTPUT_DIR, "woman_vae_latents.pt"))

print(f"man_vae_latents.pt   → {tuple(man_latents.shape)}")
print(f"woman_vae_latents.pt → {tuple(woman_latents.shape)}")
print(f"✅ Both saved to {OUTPUT_DIR}")

## 8 · Linear Interpolation in VAE Latent Space

In [ ]:
# α = 0 → man latent,  α = 1 → woman latent
# Simple lerp — no sphere projection needed (VAE latents are unconstrained)
alphas = torch.linspace(0.0, 1.0, N_INTERP)  # [N_INTERP]

interp_latents = torch.stack(
    [(1.0 - a) * man_latent + a * woman_latent for a in alphas],
    dim=0
)  # [N_INTERP, 4, 64, 64]

torch.save(interp_latents, os.path.join(OUTPUT_DIR, "interp_vae_latents.pt"))
print(f"interp_vae_latents.pt → {tuple(interp_latents.shape)}")

# Sanity: endpoints should match the originals exactly
err_start = (interp_latents[0]  - man_latent).abs().max().item()
err_end   = (interp_latents[-1] - woman_latent).abs().max().item()
print(f"Endpoint error — step 0   ↔ man   : {err_start:.2e}  (should be ~0)")
print(f"Endpoint error — step N-1 ↔ woman : {err_end:.2e}  (should be ~0)")
print("✅ Interpolation complete.")

## 9 · Decode All Interpolation Frames and Save

In [ ]:
print(f"Decoding {N_INTERP} interpolation latents...")
interp_pil = decode_latents(interp_latents, sprinter.vae, batch_size=4)
print(f"✅ Decoded {len(interp_pil)} frames.")

# Save all decoded frames
decoded_dir = os.path.join(OUTPUT_DIR, "interp_decoded")
os.makedirs(decoded_dir, exist_ok=True)
for i, img in enumerate(tqdm(interp_pil, desc="Saving frames")):
    alpha_str = f"{alphas[i].item():.3f}".replace(".", "p")
    img.save(os.path.join(decoded_dir, f"interp_{i:03d}_a{alpha_str}.png"))
print(f"✅ {len(interp_pil)} frames → {decoded_dir}")

## 10 · Contact Sheet — 10 Keyframes

In [ ]:
keyframe_idx = np.linspace(0, N_INTERP - 1, 10, dtype=int)

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle("Interpolation Contact Sheet: Masculine Man → Feminine Woman",
             fontsize=13, fontweight="bold")
for j, idx in enumerate(keyframe_idx):
    ax = axes[j // 5][j % 5]
    ax.imshow(interp_pil[idx])
    ax.set_title(f"step {idx}  α={alphas[idx]:.2f}", fontsize=9)
    ax.axis("off")
plt.tight_layout()

contact_path = os.path.join(OUTPUT_DIR, "interp_contact_sheet.png")
fig.savefig(contact_path, dpi=110, bbox_inches="tight")
plt.show()
print(f"✅ Contact sheet → {contact_path}")

## 11 · PCA of VAE Latents

In [ ]:
print("Fitting PCA on flattened VAE latents...")
all_flat  = interp_latents.reshape(N_INTERP, -1).numpy()  # [N_INTERP, 4*64*64]
pca_lat   = PCA(n_components=2)
lat_coords = pca_lat.fit_transform(all_flat)
var_lat    = pca_lat.explained_variance_ratio_.sum()
print(f"  Variance explained: {var_lat:.1%}")

plot_pca_interp(
    lat_coords, N_INTERP,
    save_path=os.path.join(OUTPUT_DIR, "interp_viz_latent_pca.png"),
    title=f"VAE Latent PCA — {N_INTERP} Interpolation Steps\nVar explained: {var_lat:.1%}",
)

## 12 · CLIP PCA — Perceptual View

In [ ]:
print("Encoding all decoded interp frames through CLIP...")
clip_embs   = encode_pil_to_clip(interp_pil, clip_model, clip_processor)
print(f"  CLIP embeddings: {clip_embs.shape}")

pca_clip    = PCA(n_components=2)
clip_coords = pca_clip.fit_transform(clip_embs.numpy())
var_clip    = pca_clip.explained_variance_ratio_.sum()
print(f"  Variance explained: {var_clip:.1%}")

plot_pca_interp(
    clip_coords, N_INTERP,
    save_path=os.path.join(OUTPUT_DIR, "interp_viz_clip_pca.png"),
    title=f"CLIP PCA (perceptual) — {N_INTERP} Interpolation Steps\nVar explained: {var_clip:.1%}",
)

## 13 · Save PCA Models + Metadata

In [ ]:
# Save fitted PCA models — run_interpolation_dps.py reuses these
with open(os.path.join(OUTPUT_DIR, "pca_latent.pkl"), "wb") as f:
    pickle.dump(pca_lat, f)
with open(os.path.join(OUTPUT_DIR, "pca_clip.pkl"), "wb") as f:
    pickle.dump(pca_clip, f)
print("✅ pca_latent.pkl and pca_clip.pkl saved.")

In [ ]:
metadata = {
    "args": {
        "output_dir":          OUTPUT_DIR,
        "n_copies":            N_COPIES,
        "n_interp":            N_INTERP,
        "controlnet_scale":    CONTROLNET_SCALE,
        "man_seed":            MAN_SEED,
        "woman_seed":          WOMAN_SEED,
        "seed":                GLOBAL_SEED,
        "controlnet_model_id": CONTROLNET_MODEL_ID,
        "sprinter_model_id":   SPRINTER_MODEL_ID,
        "architect_model_id":  ARCHITECT_MODEL_ID,
    },
    "device":                       device,
    "man_latent_shape":              list(man_latent.shape),
    "woman_latent_shape":            list(woman_latent.shape),
    "man_latent_norm":               man_latent.norm().item(),
    "woman_latent_norm":             woman_latent.norm().item(),
    "man_woman_cosine_sim_latent":   cos_sim,
    "man_woman_l2_dist_latent":      l2_dist,
    "interp_latents_shape":          list(interp_latents.shape),
    "pca_latent_var_explained":      float(var_lat),
    "pca_clip_var_explained":        float(var_clip),
    "man_prompt":                    MAN_PROMPT,
    "woman_prompt":                  WOMAN_PROMPT,
    "outputs": {
        "man_canonical":      "man_canonical.png",
        "woman_canonical":    "woman_canonical.png",
        "man_vae_latents":    "man_vae_latents.pt",
        "woman_vae_latents":  "woman_vae_latents.pt",
        "interp_vae_latents": "interp_vae_latents.pt",
        "interp_decoded":     "interp_decoded/",
        "contact_sheet":      "interp_contact_sheet.png",
        "latent_pca":         "interp_viz_latent_pca.png",
        "clip_pca":           "interp_viz_clip_pca.png",
        "base_scribble":      "base_scribble.png",
        "pca_latent_model":   "pca_latent.pkl",
        "pca_clip_model":     "pca_clip.pkl",
    },
}
with open(os.path.join(OUTPUT_DIR, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)
print("✅ metadata.json saved.")

## 14 · Summary

In [ ]:
abs_out = os.path.abspath(OUTPUT_DIR)
print("=" * 65)
print("SUMMARY")
print("=" * 65)
print(f"  Canonical man latent   : {tuple(man_latent.shape)}  norm={man_latent.norm():.3f}")
print(f"  Canonical woman latent : {tuple(woman_latent.shape)}  norm={woman_latent.norm():.3f}")
print(f"  Copies per class       : {N_COPIES}  → [{N_COPIES}, 4, 64, 64] each")
print(f"  Interpolation steps    : {N_INTERP}")
print(f"  Man↔Woman L2 (latent)  : {l2_dist:.3f}")
print(f"  Man↔Woman cos-sim      : {cos_sim:.4f}")
print(f"  VAE latent PCA var     : {var_lat:.1%}")
print(f"  CLIP PCA var           : {var_clip:.1%}")
print()
print(f"  Output dir: {abs_out}")
print()
print("  Files written:")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.isfile(fpath):
        size_kb = os.path.getsize(fpath) / 1024
        print(f"    {fname:<35}  {size_kb:7.1f} KB")
    else:
        n_files = len(os.listdir(fpath))
        print(f"    {fname:<35}  [{n_files} files]")
print("=" * 65)
print()
print("✅ Done!  Pass --targets_dir", OUTPUT_DIR, "to run_interpolation_dps.py next.")